# Visualize Results

Load results from `results/` and create clear visualizations. Run after `main.py` has produced the CSV files.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
RESULTS_DIR = os.path.join(ROOT, "results")
FIG_DIR = os.path.join(ROOT, "figures")

DATASET = "Cora"  # change if you ran main.py with another dataset

def load_results():
    """Load the three result CSVs. Returns dict with keys: rewiring, multiscale, per_node."""
    return {
        "rewiring": pd.read_csv(os.path.join(RESULTS_DIR, f"rewiring_comparison_{DATASET}.csv")),
        "multiscale": pd.read_csv(os.path.join(RESULTS_DIR, f"multiscale_hyperbolicity_{DATASET}.csv")),
        "per_node": pd.read_csv(os.path.join(RESULTS_DIR, f"per_node_analysis_{DATASET}.csv")),
    }

results = load_results()
print("Loaded:", {k: v.shape for k, v in results.items()})

## 1. Rewiring comparison

Test accuracy (mean ± std) for **original** graph vs **curvature-rewired** vs **random-rewired**.

In [ ]:
df = results["rewiring"]
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(df))
means = df["mean_test_acc"].values * 100
stds = df["std_test_acc"].values * 100
colors = ["#3498db", "#27ae60", "#e74c3c"]
bars = ax.bar(x, means, yerr=stds, capsize=6, color=colors, edgecolor="black")
ax.set_xticks(x)
ax.set_xticklabels([s.replace("_", " ").title() for s in df["strategy"]])
ax.set_ylabel("Test accuracy (%)")
ax.set_title(f"Rewiring strategies — {DATASET}")
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

## 2. Multiscale hyperbolicity

Correlation between local hyperbolicity and node accuracy at different hop radii (scales).

In [ ]:
df = results["multiscale"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

scales = df["scale"].astype(int)
corrs = df["correlation"].values
colors = ["#3498db" if c >= 0 else "#e74c3c" for c in corrs]
ax1.bar(scales.astype(str), corrs, color=colors, edgecolor="black")
ax1.axhline(0, color="black", linewidth=0.5)
ax1.set_xlabel("Scale (hop radius)")
ax1.set_ylabel("Correlation (hyperbolicity vs accuracy)")
ax1.set_title("Hyperbolicity–accuracy correlation")

ax2.bar(scales.astype(str), df["mean_hyperbolicity"].values, color="#9b59b6", edgecolor="black")
ax2.set_xlabel("Scale (hop radius)")
ax2.set_ylabel("Mean hyperbolicity")
ax2.set_title("Mean local hyperbolicity by scale")

plt.tight_layout()
plt.show()

## 3. Per-node: curvature vs accuracy

Node-level Ollivier-Ricci curvature vs whether the model predicted the node correctly. Binned view (accuracy per curvature bin) and scatter (sample).

In [ ]:
df = results["per_node"]
df["correct"] = df["correct"].astype(bool)
df["accuracy"] = df["correct"].astype(float)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: binned accuracy by curvature
n_bins = 12
df["curv_bin"] = pd.cut(df["curvature"], bins=n_bins)
bin_acc = df.groupby("curv_bin", observed=True).agg(accuracy=("accuracy", "mean"), count=("node_id", "count"))
bin_acc = bin_acc[bin_acc["count"] >= 5]  # drop tiny bins
mid = [c.mid for c in bin_acc.index]
ax1.bar(range(len(mid)), bin_acc["accuracy"].values * 100, color="#3498db", edgecolor="black")
ax1.set_xticks(range(len(mid)))
ax1.set_xticklabels([f"{m:.2f}" for m in mid], rotation=45)
ax1.set_xlabel("Curvature (bin center)")
ax1.set_ylabel("Accuracy (%)")
ax1.set_title("Accuracy by curvature bin")

# Right: scatter (subsample for readability)
sample = df.sample(n=min(400, len(df)), random_state=42)
ax2.scatter(sample["curvature"], sample["accuracy"], alpha=0.4, s=15, c=sample["accuracy"], cmap="RdYlGn")
ax2.set_xlabel("Node curvature")
ax2.set_ylabel("Correct (0/1)")
ax2.set_title("Curvature vs correct prediction (sample)")

plt.tight_layout()
plt.show()

## 4. Summary table

In [ ]:
print("Rewiring comparison:")
print(results["rewiring"][["strategy", "mean_test_acc", "std_test_acc"]].to_string(index=False))
print("\nMultiscale hyperbolicity:")
print(results["multiscale"].to_string(index=False))

## 5. Existing figures

Plots already saved by `main.py` in `figures/`:

In [ ]:
from IPython.display import Image, display

fig_files = [
    f"curvature_distribution_{DATASET}.png",
    f"curvature_vs_accuracy_{DATASET}.png",
    f"graph_curvature_{DATASET}.png",
    f"rewiring_comparison_{DATASET}.png",
    f"multiscale_hyperbolicity_{DATASET}.png",
    "training_curves.png",
]
for f in fig_files:
    path = os.path.join(FIG_DIR, f)
    if os.path.isfile(path):
        print(f)
        display(Image(path, width=500))